# EDA

## Load the data

In [1]:
import sys

sys.path.append("..")

from src.data.load_data import load_all_data

ratings, movies, users = load_all_data()

In [2]:
ratings.head(), movies.head(), users.head()

(   user_id  movie_id  rating           timestamp
 0        1      1193       5 2000-12-31 22:12:40
 1        1       661       3 2000-12-31 22:35:09
 2        1       914       3 2000-12-31 22:32:48
 3        1      3408       4 2000-12-31 22:04:35
 4        1      2355       5 2001-01-06 23:38:11,
    movie_id                               title  \
 0         1                    Toy Story (1995)   
 1         2                      Jumanji (1995)   
 2         3             Grumpier Old Men (1995)   
 3         4            Waiting to Exhale (1995)   
 4         5  Father of the Bride Part II (1995)   
 
                              genres  
 0   [Animation, Children's, Comedy]  
 1  [Adventure, Children's, Fantasy]  
 2                 [Comedy, Romance]  
 3                   [Comedy, Drama]  
 4                          [Comedy]  ,
    user_id gender  age  occupation zip_code
 0        1      F    1          10    48067
 1        2      M   56          16    70072
 2        3    

## Train/Test Split

For this project, I do not want to use a typical random train/test split. This is because it would cause the system to be training on future data and testing on past data which is not how recommendations work in real life. Because of this, I will be doing a time based split per user. This means that for each user, the model will train on their earlier interactions and test on their later interactions, simulating a situation of predicting their future behavior based on past behavior. To do this, I will be creating my own split function which you can find [here](../src/data/split.py)

In [3]:
from src.data.split import train_test_split_by_user

train_df, test_df = train_test_split_by_user(ratings, test_size=0.2)

print(train_df.shape)
print(test_df.shape)
set(train_df.index).intersection(set(test_df.index))
print(train_df["user_id"].nunique())
print(test_df["user_id"].nunique())

user = ratings["user_id"].iloc[0]

train_user = train_df[train_df["user_id"] == user]
test_user = test_df[test_df["user_id"] == user]

print(train_user[["movie_id", "rating", "timestamp"]].tail())
print(test_user[["movie_id", "rating", "timestamp"]].head())

(802553, 4)
(197656, 4)
6040
6040
    movie_id  rating           timestamp
8        594       4 2000-12-31 22:37:48
12      2398       4 2000-12-31 22:38:01
28      1545       4 2001-01-06 23:35:39
23       527       5 2001-01-06 23:36:35
10       595       5 2001-01-06 23:37:48
    movie_id  rating           timestamp
16      2687       3 2001-01-06 23:37:48
29       745       3 2001-01-06 23:37:48
33       588       4 2001-01-06 23:37:48
40         1       5 2001-01-06 23:37:48
4       2355       5 2001-01-06 23:38:11


## Popularity Recommender

The next thing that I need to do is to build a popularity recommender system. This system should take into account not only the average rating, but also how frequently it is rated. These two metrics will be combined into a popularity score which will be used when making recommendations. You can find the model [here](../models/popularity.py). I have implemented multiple approaches to calculating the popularity score (linear, log, bayesian). Below you can see the outputs of each

In [4]:
from src.models.popularity import PopularityRecommender

pop_model = PopularityRecommender(min_ratings=50)
pop_model.fit(train_df)

pop_model.recommend(user_id=1, ratings=train_df, k=10)

,movie_id,avg_rating,num_ratings,score
2623,2858,4.328607,3174,4.328607
1092,1196,4.295207,2754,3.726843
1106,1210,4.024473,2656,3.367675
1094,1198,4.474119,2299,3.240706
2348,2571,4.332335,2338,3.191241
575,593,4.362924,2298,3.158790
571,589,4.081886,2418,3.109641
106,110,4.241599,2202,2.942659
791,858,4.528770,2016,2.876497
464,480,3.760482,2409,2.854127


In [5]:
pop_model = PopularityRecommender(min_ratings=50, method="log")
pop_model.fit(train_df)

pop_model.recommend(user_id=1, ratings=train_df, k=10)

,movie_id,avg_rating,num_ratings,score
2623,2858,4.328607,3174,34.901834
308,318,4.565570,1975,34.647331
1094,1198,4.474119,2299,34.632655
791,858,4.528770,2016,34.461070
1092,1196,4.295207,2754,34.023076
575,593,4.362924,2298,33.770035
2348,2571,4.332335,2338,33.607999
49,50,4.533943,1532,33.256387
106,110,4.241599,2202,32.650024
286,296,4.278749,1887,32.275776


In [6]:
pop_model = PopularityRecommender(min_ratings=50, method="bayesian")
pop_model.fit(train_df)

pop_model.recommend(user_id=1, ratings=train_df, k=10)

,movie_id,avg_rating,num_ratings,score
308,318,4.565570,1975,4.536421
791,858,4.528770,2016,4.501091
49,50,4.533943,1532,4.497632
1819,2019,4.582857,525,4.478702
1052,1148,4.521411,794,4.454092
1094,1198,4.474119,2299,4.450938
699,745,4.534722,576,4.442897
704,750,4.474459,1155,4.429256
831,904,4.483213,834,4.421101
839,912,4.402740,1460,4.369042


Now we can run all three methods again and merge it with the movies to see the difference in recommendations based on the method

In [7]:
for method in ["simple", "log", "bayesian"]:
    model = PopularityRecommender(method=method)
    model.fit(train_df)
    
    recs = model.recommend(user_id=1, ratings=train_df, k=5)
    recs = recs.merge(movies, on="movie_id")
    
    print(f"\nMethod: {method}")
    print(recs[["title", "score"]])


Method: simple
                                               title     score
0                             American Beauty (1999)  4.328607
1  Star Wars: Episode V - The Empire Strikes Back...  3.726843
2  Star Wars: Episode VI - Return of the Jedi (1983)  3.367675
3                     Raiders of the Lost Ark (1981)  3.240706
4                                 Matrix, The (1999)  3.191241

Method: log
                                               title      score
0                             American Beauty (1999)  34.901834
1                   Shawshank Redemption, The (1994)  34.647331
2                     Raiders of the Lost Ark (1981)  34.632655
3                              Godfather, The (1972)  34.461070
4  Star Wars: Episode V - The Empire Strikes Back...  34.023076

Method: bayesian
                                               title     score
0                   Shawshank Redemption, The (1994)  4.536421
1                              Godfather, The (1972)  4.501091
2 

### Evaluate the Popularity Recommender

Now that we have a working popularity recommender, we need to evaluate this system by checking of the $K$ movies suggested, how many of them are actually present in the user's test set? For this, I will use Precision@K, Recall@K, and NCDG@K which you can find [here](../src/evaluation/metrics.py). This metric will be used to evaluate the model [here](../src/evaluation/evaluate.py)

In [8]:
from src.evaluation.evaluate import evaluate_model 

for method in ["simple", "log", "bayesian"]:
    model = PopularityRecommender(min_ratings=50, method=method)
    model.fit(train_df)

    results = evaluate_model(model, train_df, test_df, k=10)

    print(f"\nMethod: {method}")
    print(results)


Method: simple
{'precision_at_k': 0.09995033112582782, 'recall_at_k': 0.03654853059243537, 'ndcg_at_k': 0.10832719465327392}

Method: log
{'precision_at_k': 0.09274834437086094, 'recall_at_k': 0.0356066723773692, 'ndcg_at_k': 0.09865172816438679}

Method: bayesian
{'precision_at_k': 0.05440397350993378, 'recall_at_k': 0.01975310850356312, 'ndcg_at_k': 0.058728381652880156}


## Content Recommender

Now I want to build a content recommender to recommend movies similar to the ones that a given user has liked in the past. This will allow recommendations to be more personalized rather than giving everyone the same recommendations using the popularity recommender. Eventually, the combination of these recommenders will be used to recommend popular movies that match the given user's preferences.

In [9]:
from src.features.content import build_genre_matrix
from src.models.content import ContentRecommender

genre_matrix = build_genre_matrix(movies)

content_model = ContentRecommender()
content_model.fit(movies, genre_matrix)

recs = content_model.recommend(user_id=1, ratings=train_df, k=10)
recs = recs.merge(movies, on="movie_id")

recs[["title", "score"]]

,title,score
0,Watership Down (1978),1.081081
1,Wide Awake (1998),1.027027
2,Babe (1995),1.027027
3,Pollyanna (1960),1.027027
4,This Is Spinal Tap (1984),0.972973
5,"Far Off Place, A (1993)",0.972973
6,Hercules (1997),0.918919
7,Free Willy 3: The Rescue (1997),0.918919
8,Lethal Weapon (1987),0.918919
9,Lethal Weapon 2 (1989),0.918919


### Evaluate the Content Recommender

Now that we have a working content recommender, we can evaluate it using the same methods as the popularity recommender.

In [10]:
results = evaluate_model(content_model, train_df, test_df, k=10)
print(results)

{'precision_at_k': 0.035546357615894045, 'recall_at_k': 0.01703134401883876, 'ndcg_at_k': 0.03748604687594514}


## Collaborative Recommender

Now that we have both a popularity and content recommender, I will build a collaborative recommender. This will find user interaction patterns to recommend movies. It will find patterns like "Users who like movie A also like movie B" and make recommendations based on that.

In [11]:
from src.models.item_collaborative import ItemCollaborativeRecommender

item_cf_model = ItemCollaborativeRecommender()
item_cf_model.fit(train_df)

recs = item_cf_model.recommend(user_id=1, ratings=train_df, k=10)
recs = recs.merge(movies, on="movie_id")

recs[["title", "score"]]

,title,score
0,Star Wars: Episode V - The Empire Strikes Back...,61.622931
1,Raiders of the Lost Ark (1981),59.025120
2,"Shawshank Redemption, The (1994)",58.605866
3,Toy Story (1995),57.299281
4,"Silence of the Lambs, The (1991)",57.243673
5,Groundhog Day (1993),56.619282
6,Pulp Fiction (1994),56.357849
7,Star Wars: Episode VI - Return of the Jedi (1983),56.105570
8,Ghostbusters (1984),56.010108
9,Stand by Me (1986),55.010079


### Evaluate the Collaborative Recommender

Now that we have a working collaborative recommender, we can evaluate it using the same metrics as before

In [12]:
results = evaluate_model(item_cf_model, train_df, test_df, k=10)
print(results)

{'precision_at_k': 0.1318708609271523, 'recall_at_k': 0.06413266516779224, 'ndcg_at_k': 0.147047384007298}


# Compare Models

Now we want to compare each of the models side by side, and then work on combining all of them into a hybrid recommender.

In [13]:
import pandas as pd

models = {
    "popularity_simple": PopularityRecommender(method="simple").fit(train_df),
    "popularity_log": PopularityRecommender(method="log").fit(train_df),
    "popularity_bayesian": PopularityRecommender(method="bayesian").fit(train_df),
    "content": ContentRecommender().fit(movies, genre_matrix),
    "item_collaborative": ItemCollaborativeRecommender().fit(train_df)
}

results = []

for name, model in models.items():
    metrics = evaluate_model(model, train_df, test_df, k=10)
    metrics["model"] = name
    results.append(metrics)

results_df = pd.DataFrame(results)
results_df = results_df[["model", "precision_at_k", "recall_at_k", "ndcg_at_k"]]
print(results_df)
    

                 model  precision_at_k  recall_at_k  ndcg_at_k
0    popularity_simple        0.099950     0.036549   0.108327
1       popularity_log        0.092748     0.035607   0.098652
2  popularity_bayesian        0.054404     0.019753   0.058728
3              content        0.035546     0.017031   0.037486
4   item_collaborative        0.131871     0.064133   0.147047


From this table above we see that each model on its own performs pretty poorly. The goal of making all of these models was to eventually combine them into a single hybrid model to hopefully achieve better performance

# Hybrid Model

The hybrid model will be combination of the three models made above to hopefully achieve a more performant model. This will be done by taking the recommendations from each model, along with a weight assigned to each model, to give each recommendation a hybrid score. This hybrid score will be used to make the recommendations

In [14]:
from src.models.hybrid import HybridRecommender

hybrid_model = HybridRecommender(
    PopularityRecommender(method="bayesian"),
    ContentRecommender(),
    ItemCollaborativeRecommender(min_rating=4),
    weights={
        "popularity": 0.15,
        "content": 0.25,
        "collaborative": 0.60,
    }
)

hybrid_model.fit(train_df, movies, genre_matrix)

recs = hybrid_model.recommend(user_id=1, ratings=train_df, k=10)
recs = recs.merge(movies, on="movie_id")

recs[["title", "score"]]

,title,score
0,Star Wars: Episode V - The Empire Strikes Back...,0.749910
1,"Shawshank Redemption, The (1994)",0.633925
2,Raiders of the Lost Ark (1981),0.619839
3,"Silence of the Lambs, The (1991)",0.512831
4,Babe (1995),0.480721
5,Pulp Fiction (1994),0.448204
6,Toy Story (1995),0.436250
7,"Godfather, The (1972)",0.419894
8,Groundhog Day (1993),0.407496
9,American Beauty (1999),0.397740


### Evaluate the Hybrid Recommender

Now that we ave the model, we can evaluate it using the same metrics as before

In [15]:
results = evaluate_model(hybrid_model, train_df, test_df, k=10)
print(results)

{'precision_at_k': 0.12847682119205298, 'recall_at_k': 0.06188960406979654, 'ndcg_at_k': 0.14171043534972558}


### Compare weights

We can also compare different weights to see what setup yields the best evaluation results

In [16]:
weight_configs = {
    "collab_heavy": {
        "popularity": 0.15,
        "content": 0.25,
        "collaborative": 0.60,
    },
    "balanced": {
        "popularity": 0.33,
        "content": 0.33,
        "collaborative": 0.34,
    },
    "content_heavy": {
        "popularity": 0.15,
        "content": 0.60,
        "collaborative": 0.25,
    },
}

hybrid_results = []

for name, weights in weight_configs.items():
    model = HybridRecommender(
        popularity_model=PopularityRecommender(method="bayesian"),
        content_model=ContentRecommender(),
        collaborative_model=ItemCollaborativeRecommender(min_rating=4),
        weights=weights,
    )

    model.fit(train_df, movies, genre_matrix)
    metrics = evaluate_model(model, train_df, test_df, k=10)
    metrics["model"] = name
    hybrid_results.append(metrics)

pd.DataFrame(hybrid_results)

,precision_at_k,recall_at_k,ndcg_at_k,model
0,0.128477,0.061890,0.141710,collab_heavy
1,0.103940,0.046954,0.115395,balanced
2,0.050232,0.025675,0.057983,content_heavy
